#### 1) First page 

In [4]:
from bs4 import BeautifulSoup
import pandas as pd
import requests

# Base URL for Quotes to Scrape
url = "http://quotes.toscrape.com/"
response = requests.get(url)

# Parse the HTML content
soup = BeautifulSoup(response.text, "html.parser")

# Find all quote containers on the page
quote_boxes = soup.find_all("div", class_="quote")

quotes_data = []

for box in quote_boxes:
  text = box.find("span", class_="text").get_text()
  author = box.find("small", class_="author").get_text()
  tags = [tag.get_text() for tag in box.find_all("a", class_="tag")]

  quotes_data.append({"Quote": text, "Author": author, "Tags": ", ".join(tags)})

# Convert into a Pandas DataFrame
df = pd.DataFrame(quotes_data)
df

,Quote,Author,Tags
0,“The world as we have created it is a process ...,Albert Einstein,"change, deep-thoughts, thinking, world"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities, choices"
2,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy, books, classic, humor"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself, inspirational"
5,“Try not to become a man of success. Rather be...,Albert Einstein,"adulthood, success, value"
6,“It is better to be hated for what you are tha...,André Gide,"life, love"
7,"“I have not failed. I've just found 10,000 way...",Thomas A. Edison,"edison, failure, inspirational, paraphrased"
8,“A woman is like a tea bag; you never know how...,Eleanor Roosevelt,misattributed-eleanor-roosevelt
9,"“A day without sunshine is like, you know, nig...",Steve Martin,"humor, obvious, simile"


#### 2) Scraping Multiple Pages (Pagination)
Because quotes span across multiple pages, you can write a loop that automatically navigates through them until no more pages remain:

In [6]:
all_quotes = []
page = 1

while True:
  url = f"http://quotes.toscrape.com/page/{page}/"
  response = requests.get(url)

  # If the page doesn't exist (e.g., 404 error), break the loop
  if response.status_code != 200:
    break

  soup = BeautifulSoup(response.text, "html.parser")
  quote_boxes = soup.find_all("div", class_="quote")

  for box in quote_boxes:
    text = box.find("span", class_="text").get_text()
    author = box.find("small", class_="author").get_text()
    tags = [tag.get_text() for tag in box.find_all("a", class_="tag")]
    all_quotes.append({"Quote": text, "Author": author, "Tags": ", ".join(tags)})

  # Check if a "Next" button exists; if not, you've reached the last page
  next_btn = soup.find("li", class_="next")
  if not next_btn:
    break

  page += 1

# Convert full multi-page dataset to DataFrame and save
df_all = pd.DataFrame(all_quotes)
df_all.to_csv("quotes_scraped.csv", index=False, encoding="utf-8")
print(f"Successfully scraped {len(df_all)} quotes across {page} pages!")
df_all

Successfully scraped 100 quotes across 10 pages!


,Quote,Author,Tags
0,“The world as we have created it is a process ...,Albert Einstein,"change, deep-thoughts, thinking, world"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities, choices"
2,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy, books, classic, humor"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself, inspirational"
...,...,...,...
95,“You never really understand a person until yo...,Harper Lee,better-life-empathy
96,“You have to write the book that wants to be w...,Madeleine L'Engle,"books, children, difficult, grown-ups, write, ..."
97,“Never tell the truth to people who are not wo...,Mark Twain,truth
98,"“A person's a person, no matter how small.”",Dr. Seuss,inspirational
